In [1]:
# ============================================
# الخلية 1: تثبيت المكتبات بشكل صحيح
# ============================================

import subprocess
import sys

print("="*60)
print("📦 تثبيت المكتبات المطلوبة")
print("="*60)

# قائمة المكتبات المطلوبة
packages = [
    "ultralytics",
    "opencv-python",
    "matplotlib",
    "numpy",
    "tqdm",
    "pyyaml",
    "wget",
    "patool"
]

# تثبيت كل مكتبة
for package in packages:
    print(f"\n⏳ تثبيت: {package}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print(f"✅ تم تثبيت {package}")

# تثبيت أدوات النظام
!apt-get update -qq
!apt-get install -y unrar p7zip-full -qq

print("\n✅ تم تثبيت جميع المكتبات بنجاح!")

# الآن استيراد المكتبات
import os
import torch
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import random
import shutil
from datetime import datetime
import zipfile
import patoolib
from tqdm import tqdm
from collections import Counter
import wget

print(f"\n🎮 GPU متاح: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   • GPU: {torch.cuda.get_device_name(0)}")
    print(f"   • الذاكرة: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

📦 تثبيت المكتبات المطلوبة

⏳ تثبيت: ultralytics
✅ تم تثبيت ultralytics

⏳ تثبيت: opencv-python
✅ تم تثبيت opencv-python

⏳ تثبيت: matplotlib
✅ تم تثبيت matplotlib

⏳ تثبيت: numpy
✅ تم تثبيت numpy

⏳ تثبيت: tqdm
✅ تم تثبيت tqdm

⏳ تثبيت: pyyaml
✅ تم تثبيت pyyaml

⏳ تثبيت: wget
✅ تم تثبيت wget

⏳ تثبيت: patool
✅ تم تثبيت patool
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

✅ تم تثبيت جميع المكتبات بنجاح!
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🎮 GPU متاح: False


In [2]:
# ============================================
# الخلية 2: فك ضغط Self-Driving Cars Dataset
# ============================================

print("="*60)
print("🚗 فك ضغط Self-Driving Cars Dataset")
print("="*60)

# مسار الملف المضغوط
car_rar_path = '/content/car.rar'
car_extract_path = '/content/car_dataset'

# التحقق من وجود الملف
if not os.path.exists(car_rar_path):
    print(f"\n❌ الملف غير موجود: {car_rar_path}")
    print("🔍 البحث عن الملف...")
    !find /content -name "*.rar" -type f
    found = !find /content -name "*.rar" -type f | head -1
    if found:
        car_rar_path = found[0]
        print(f"✅ تم العثور على: {car_rar_path}")
    else:
        raise Exception("لم يتم العثور على ملف car.rar")

print(f"\n📦 ملف RAR: {car_rar_path}")
print(f"📂 مجلد الاستخراج: {car_extract_path}")

# إنشاء مجلد الاستخراج
os.makedirs(car_extract_path, exist_ok=True)

# فك الضغط
try:
    print("\n⏳ جاري فك الضغط...")
    patoolib.extract_archive(car_rar_path, outdir=car_extract_path)
    print("✅ تم فك الضغط بنجاح!")
except Exception as e:
    print(f"⚠️ فشل patool: {e}")
    !unrar x {car_rar_path} {car_extract_path}/ -y
    print("✅ تم فك الضغط باستخدام unrar!")

INFO patool: Extracting /content/car.rar ...
INFO patool: running /usr/bin/unrar x -kb -or -- /content/car.rar


🚗 فك ضغط Self-Driving Cars Dataset

📦 ملف RAR: /content/car.rar
📂 مجلد الاستخراج: /content/car_dataset

⏳ جاري فك الضغط...


INFO patool: ... /content/car.rar extracted to `/content/car_dataset'.


✅ تم فك الضغط بنجاح!


In [6]:
# ============================================
# الخلية 3 المصححة: تحميل GTSDB (باستخدام رابط sid.erda.dk)
# ============================================

print("="*60)
print("🇩🇪 تحميل GTSDB (German Traffic Sign Detection Benchmark)")
print("="*60)

gtsdb_zip_path = '/content/GTSDB.zip'
gtsdb_extract_path = '/content/gtsdb_dataset'

if not os.path.exists(gtsdb_zip_path):
    print("\n⏳ جاري تحميل GTSDB (1.5 GB) من الرابط البديل...")
    # الرابط الصحيح الذي جربته وعمل
    url = "https://sid.erda.dk/public/archives/ff17dc924eba88d5d01a807357d6614c/FullIJCNN2013.zip"
    # استخدام wget مباشرة لأنه أكثر استقراراً في بعض الأحيان
    !wget -O {gtsdb_zip_path} {url}
    print("\n✅ تم التحميل!")
else:
    print(f"\n✅ GTSDB.zip موجود بالفعل")

# فك ضغط GTSDB (كما هو)
print("\n⏳ جاري فك ضغط GTSDB...")
os.makedirs(gtsdb_extract_path, exist_ok=True)
# تأكد من استخدام المسار الصحيح للملف المضغوط
!unzip -q {gtsdb_zip_path} -d {gtsdb_extract_path}
print("✅ تم فك الضغط!")

# البحث عن gt.txt (هذا الجزء يبقى كما هو)
gt_file = None
gtsdb_img_dir = None
print("\n🔍 البحث عن ملف gt.txt...")
for root, dirs, files in os.walk(gtsdb_extract_path):
    if 'gt.txt' in files:
        gt_file = os.path.join(root, 'gt.txt')
        gtsdb_img_dir = root
        print(f"✅ تم العثور على gt.txt في: {gt_file}")
        break

if not gt_file:
    raise Exception("لم يتم العثور على gt.txt في GTSDB")
else:
    print(f"📂 مجلد الصور: {gtsdb_img_dir}")

🇩🇪 تحميل GTSDB (German Traffic Sign Detection Benchmark)

⏳ جاري تحميل GTSDB (1.5 GB) من الرابط البديل...
--2026-03-03 13:28:56--  https://sid.erda.dk/public/archives/ff17dc924eba88d5d01a807357d6614c/FullIJCNN2013.zip
Resolving sid.erda.dk (sid.erda.dk)... 130.225.104.13
Connecting to sid.erda.dk (sid.erda.dk)|130.225.104.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1661866983 (1.5G) [application/zip]
Saving to: ‘/content/GTSDB.zip’

/content/GTSDB.zip  100%[===================>]   1.55G  33.9MB/s    in 63s     

2026-03-03 13:29:59 (25.3 MB/s) - ‘/content/GTSDB.zip’ saved [1661866983/1661866983]


✅ تم التحميل!

⏳ جاري فك ضغط GTSDB...
✅ تم فك الضغط!

🔍 البحث عن ملف gt.txt...
✅ تم العثور على gt.txt في: /content/gtsdb_dataset/FullIJCNN2013/gt.txt
📂 مجلد الصور: /content/gtsdb_dataset/FullIJCNN2013


In [7]:
# ============================================
# الخلية 4: تحليل بنية Self-Driving Cars Dataset
# ============================================

def analyze_car_structure():
    """تحليل بنية مجلدات Self-Driving Cars"""

    print("="*60)
    print("🔍 تحليل بنية Self-Driving Cars Dataset")
    print("="*60)

    splits = {}

    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(car_extract_path, split, 'images')
        label_dir = os.path.join(car_extract_path, split, 'labels')

        if os.path.exists(img_dir):
            images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
            labels = os.listdir(label_dir) if os.path.exists(label_dir) else []

            splits[split] = {
                'images': img_dir,
                'labels': label_dir if os.path.exists(label_dir) else None,
                'num_images': len(images),
                'num_labels': len(labels)
            }

            print(f"\n📂 {split.upper()}:")
            print(f"   • الصور: {len(images)}")
            print(f"   • العلامات: {len(labels)}")
            print(f"   • مجلد الصور: {img_dir}")

    return splits

car_splits = analyze_car_structure()

🔍 تحليل بنية Self-Driving Cars Dataset

📂 TRAIN:
   • الصور: 3530
   • العلامات: 3530
   • مجلد الصور: /content/car_dataset/train/images

📂 VALID:
   • الصور: 801
   • العلامات: 801
   • مجلد الصور: /content/car_dataset/valid/images

📂 TEST:
   • الصور: 638
   • العلامات: 638
   • مجلد الصور: /content/car_dataset/test/images


In [8]:
# ============================================
# الخلية 5: قراءة علامات GTSDB
# ============================================

def read_gtsdb_annotations(gt_file):
    """قراءة علامات GTSDB من ملف gt.txt"""
    annotations = {}

    print("="*60)
    print("📖 قراءة علامات GTSDB")
    print("="*60)

    with open(gt_file, 'r') as f:
        for line in tqdm(f, desc="معالجة GTSDB"):
            parts = line.strip().split(';')
            if len(parts) >= 6:
                img_name = parts[0]
                xmin = int(parts[1])
                ymin = int(parts[2])
                xmax = int(parts[3])
                ymax = int(parts[4])
                class_id = int(parts[5])

                if img_name not in annotations:
                    annotations[img_name] = []

                annotations[img_name].append({
                    'bbox': [xmin, ymin, xmax-xmin, ymax-ymin],
                    'class_id': class_id  # 0-42
                })

    print(f"\n📸 صور GTSDB: {len(annotations)}")
    total_signs = sum(len(v) for v in annotations.values())
    print(f"🚦 إجمالي الإشارات: {total_signs}")

    return annotations

gtsdb_annotations = read_gtsdb_annotations(gt_file)

📖 قراءة علامات GTSDB


معالجة GTSDB: 1213it [00:00, 250872.33it/s]


📸 صور GTSDB: 741
🚦 إجمالي الإشارات: 1213


In [9]:
# ============================================
# الخلية 6: دمج الفئات
# ============================================

# فئات GTSDB (43 فئة)
gtsdb_classes = [f'GTSDB_Class_{i:02d}' for i in range(43)]

# فئات Self-Driving Cars (15 فئة)
car_classes = [
    'Green Light', 'Red Light', 'Speed Limit 10', 'Speed Limit 100',
    'Speed Limit 110', 'Speed Limit 120', 'Speed Limit 20', 'Speed Limit 30',
    'Speed Limit 40', 'Speed Limit 50', 'Speed Limit 60', 'Speed Limit 70',
    'Speed Limit 80', 'Speed Limit 90', 'Stop'
]

# دمج الفئات
all_classes = gtsdb_classes + car_classes
num_classes = len(all_classes)

print("="*60)
print("📝 دمج الفئات")
print("="*60)
print(f"\n📊 إجمالي الفئات: {num_classes}")
print(f"   • GTSDB: 43 فئة (0-42)")
print(f"   • Self-Driving Cars: 15 فئة (43-57)")

# إنشاء mapping
class_mapping = {i: name for i, name in enumerate(all_classes)}
print(f"\n📋 أول 10 فئات:")
for i in range(10):
    print(f"   • {i}: {class_mapping[i]}")

📝 دمج الفئات

📊 إجمالي الفئات: 58
   • GTSDB: 43 فئة (0-42)
   • Self-Driving Cars: 15 فئة (43-57)

📋 أول 10 فئات:
   • 0: GTSDB_Class_00
   • 1: GTSDB_Class_01
   • 2: GTSDB_Class_02
   • 3: GTSDB_Class_03
   • 4: GTSDB_Class_04
   • 5: GTSDB_Class_05
   • 6: GTSDB_Class_06
   • 7: GTSDB_Class_07
   • 8: GTSDB_Class_08
   • 9: GTSDB_Class_09


In [10]:
# ============================================
# الخلية 7: إنشاء مجلد YOLO موحد
# ============================================

def convert_to_yolo_format(x, y, width, height, img_width, img_height):
    """تحويل إحداثيات لصيغة YOLO"""
    x_center = (x + width/2) / img_width
    y_center = (y + height/2) / img_height
    w_norm = width / img_width
    h_norm = height / img_height

    x_center = max(0, min(1, x_center))
    y_center = max(0, min(1, y_center))
    w_norm = max(0, min(1, w_norm))
    h_norm = max(0, min(1, h_norm))

    return x_center, y_center, w_norm, h_norm

# إنشاء مجلد موحد
merged_dir = '/content/merged_dataset'
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(merged_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(merged_dir, 'labels', split), exist_ok=True)

print("="*60)
print("🔄 إنشاء مجلد YOLO موحد")
print("="*60)
print(f"📂 المسار: {merged_dir}")

# دمج GTSDB (كلها في train)
print("\n⏳ دمج GTSDB...")
gtsdb_added = 0
for img_name, annotations in tqdm(gtsdb_annotations.items(), desc="GTSDB"):
    img_path = os.path.join(gtsdb_img_dir, img_name)
    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]

    # حفظ الصورة
    new_img_name = f"gtsdb_{img_name.replace('.ppm', '.jpg')}"
    dst_img = os.path.join(merged_dir, 'images', 'train', new_img_name)
    cv2.imwrite(dst_img, img)

    # حفظ العلامات
    label_file = os.path.join(merged_dir, 'labels', 'train', new_img_name.replace('.jpg', '.txt'))

    with open(label_file, 'w') as f:
        for ann in annotations:
            x, y, bw, bh = ann['bbox']
            class_id = ann['class_id']  # 0-42

            xc, yc, wn, hn = convert_to_yolo_format(x, y, bw, bh, w, h)
            f.write(f"{class_id} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

    gtsdb_added += 1

print(f"✅ تمت إضافة {gtsdb_added} صورة GTSDB")

# دمج Self-Driving Cars
print("\n⏳ دمج Self-Driving Cars...")
car_added = 0
car_skipped = 0

for split_name, split_info in car_splits.items():
    img_dir = split_info['images']
    label_dir = split_info['labels']

    # تحديد المجلد الوجهة
    if split_name == 'train':
        dest_split = 'train'
    elif split_name == 'valid':
        dest_split = 'val'
    else:
        dest_split = 'test'

    if not os.path.exists(img_dir):
        continue

    images = os.listdir(img_dir)

    for img_name in tqdm(images, desc=f"Self-Driving {split_name}"):
        img_path = os.path.join(img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            car_skipped += 1
            continue

        h, w = img.shape[:2]

        # البحث عن ملف العلامات
        label_name = img_name.replace('.jpg', '.txt').replace('.png', '.txt')
        label_path = os.path.join(label_dir, label_name) if label_dir else None

        if not label_path or not os.path.exists(label_path):
            car_skipped += 1
            continue

        # حفظ الصورة
        new_img_name = f"car_{img_name}"
        dst_img = os.path.join(merged_dir, 'images', dest_split, new_img_name)
        cv2.imwrite(dst_img, img)

        # حفظ العلامات (مع إزاحة class_id +43)
        label_file = os.path.join(merged_dir, 'labels', dest_split, new_img_name.replace('.jpg', '.txt'))

        with open(label_file, 'w') as f:
            with open(label_path, 'r') as label_f:
                for line in label_f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = int(parts[0]) + 43  # إزاحة
                        xc, yc, wn, hn = map(float, parts[1:5])
                        f.write(f"{class_id} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

        car_added += 1

print(f"✅ تمت إضافة {car_added} صورة Self-Driving Cars")
print(f"⚠️ تم تخطي {car_skipped} صورة")

🔄 إنشاء مجلد YOLO موحد
📂 المسار: /content/merged_dataset

⏳ دمج GTSDB...


GTSDB: 100%|██████████| 741/741 [00:08<00:00, 87.19it/s]


✅ تمت إضافة 741 صورة GTSDB

⏳ دمج Self-Driving Cars...


Self-Driving test: 100%|██████████| 638/638 [00:02<00:00, 280.97it/s]

✅ تمت إضافة 4969 صورة Self-Driving Cars
⚠️ تم تخطي 0 صورة


In [11]:
# ============================================
# الخلية 8: إنشاء ملف data.yaml
# ============================================

data_config = {
    'path': merged_dir,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': num_classes,
    'names': all_classes
}

yaml_path = '/content/merged_dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("="*60)
print("📝 ملف data.yaml للمجموعة المدمجة")
print("="*60)
print(f"\n📁 المسار: {yaml_path}")
print(f"\n📋 المحتوى:")
print("-"*40)
!cat {yaml_path}

📝 ملف data.yaml للمجموعة المدمجة

📁 المسار: /content/merged_dataset.yaml

📋 المحتوى:
----------------------------------------
names:
- GTSDB_Class_00
- GTSDB_Class_01
- GTSDB_Class_02
- GTSDB_Class_03
- GTSDB_Class_04
- GTSDB_Class_05
- GTSDB_Class_06
- GTSDB_Class_07
- GTSDB_Class_08
- GTSDB_Class_09
- GTSDB_Class_10
- GTSDB_Class_11
- GTSDB_Class_12
- GTSDB_Class_13
- GTSDB_Class_14
- GTSDB_Class_15
- GTSDB_Class_16
- GTSDB_Class_17
- GTSDB_Class_18
- GTSDB_Class_19
- GTSDB_Class_20
- GTSDB_Class_21
- GTSDB_Class_22
- GTSDB_Class_23
- GTSDB_Class_24
- GTSDB_Class_25
- GTSDB_Class_26
- GTSDB_Class_27
- GTSDB_Class_28
- GTSDB_Class_29
- GTSDB_Class_30
- GTSDB_Class_31
- GTSDB_Class_32
- GTSDB_Class_33
- GTSDB_Class_34
- GTSDB_Class_35
- GTSDB_Class_36
- GTSDB_Class_37
- GTSDB_Class_38
- GTSDB_Class_39
- GTSDB_Class_40
- GTSDB_Class_41
- GTSDB_Class_42
- Green Light
- Red Light
- Speed Limit 10
- Speed Limit 100
- Speed Limit 110
- Speed Limit 120
- Speed Limit 20
- Speed Limit 30
- Spe

In [12]:
# ============================================
# الخلية 9: إحصائيات المجموعة المدمجة
# ============================================

print("="*60)
print("📊 إحصائيات المجموعة المدمجة")
print("="*60)

total_train = len(os.listdir(os.path.join(merged_dir, 'images', 'train'))) if os.path.exists(os.path.join(merged_dir, 'images', 'train')) else 0
total_val = len(os.listdir(os.path.join(merged_dir, 'images', 'val'))) if os.path.exists(os.path.join(merged_dir, 'images', 'val')) else 0
total_test = len(os.listdir(os.path.join(merged_dir, 'images', 'test'))) if os.path.exists(os.path.join(merged_dir, 'images', 'test')) else 0

print(f"\n📂 TRAIN:")
print(f"   • الصور: {total_train}")

print(f"\n📂 VAL:")
print(f"   • الصور: {total_val}")

print(f"\n📂 TEST:")
print(f"   • الصور: {total_test}")

print(f"\n✅ الإجمالي: {total_train + total_val + total_test} صورة")
print(f"📊 عدد الفئات: {num_classes}")

📊 إحصائيات المجموعة المدمجة

📂 TRAIN:
   • الصور: 4271

📂 VAL:
   • الصور: 801

📂 TEST:
   • الصور: 638

✅ الإجمالي: 5710 صورة
📊 عدد الفئات: 58


In [ ]:
# ============================================
# الخلية 10: تدريب YOLO مع حفظ دوري وتقييم
# ============================================

import os
import torch
from ultralytics import YOLO
from datetime import datetime
import time

print("="*60)
print("🚀 بدء تدريب YOLOv8 على المجموعة المدمجة (مع حفظ دوري)")
print("="*60)

# معلومات GPU
if torch.cuda.is_available():
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   • الذاكرة: {total_memory:.1f} GB")

    # اختيار النموذج حسب الذاكرة
    if total_memory > 20:
        model = YOLO('yolov8x.pt')
        batch_size = 32
        print(f"✅ استخدام YOLOv8x")
    else:
        model = YOLO('yolov8m.pt')
        batch_size = 16
        print(f"✅ استخدام YOLOv8m")
else:
    model = YOLO('yolov8n.pt')
    batch_size = 8
    print(f"⚠️ استخدام YOLOv8n على CPU")

# تحديد مجلد الحفظ
project_dir = '/content/merged_training'
save_dir = os.path.join(project_dir, 'yolov8_merged')
os.makedirs(save_dir, exist_ok=True)

print(f"\n⚙️ إعدادات التدريب:")
print(f"   • حجم الدفعة: {batch_size}")
print(f"   • حجم الصورة: 640")
print(f"   • عدد الفئات: {num_classes}")
print(f"   • مجلد الحفظ: {save_dir}")

# دالة لحفظ النموذج مع timestamp
def save_model_checkpoint(model, epoch, metrics=None):
    """حفظ النموذج مع معلومات التقييم"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    # حفظ النموذج
    checkpoint_name = f'yolov8_merged_epoch{epoch}_{timestamp}.pt'
    checkpoint_path = os.path.join(project_dir, checkpoint_name)
    model.save(checkpoint_path)

    print(f"\n✅ تم حفظ النموذج بعد Epoch {epoch}:")
    print(f"   • المسار: {checkpoint_path}")
    print(f"   • الحجم: {os.path.getsize(checkpoint_path) / 1e6:.2f} MB")

    # حفظ معلومات التقييم إذا وجدت
    if metrics:
        metrics_file = os.path.join(project_dir, f'metrics_epoch{epoch}_{timestamp}.txt')
        with open(metrics_file, 'w') as f:
            f.write(f"Epoch: {epoch}\n")
            f.write(f"mAP50: {metrics.box.map50:.4f}\n")
            f.write(f"mAP50-95: {metrics.box.map:.4f}\n")
            f.write(f"Precision: {metrics.box.mp:.4f}\n")
            f.write(f"Recall: {metrics.box.mr:.4f}\n")
        print(f"   • تم حفظ المقاييس في: {metrics_file}")

    return checkpoint_path

# التحقق من وجود نموذج سابق لاستئناف التدريب
resume_path = None
latest_checkpoint = None

# البحث عن آخر checkpoint
checkpoints = [f for f in os.listdir(project_dir) if f.startswith('yolov8_merged_epoch') and f.endswith('.pt')]
if checkpoints:
    # استخراج رقم Epoch من اسم الملف
    epochs = []
    for cp in checkpoints:
        try:
            epoch_num = int(cp.split('epoch')[1].split('_')[0])
            epochs.append((epoch_num, cp))
        except:
            continue

    if epochs:
        epochs.sort()
        latest_epoch, latest_checkpoint = epochs[-1]
        resume_path = os.path.join(project_dir, latest_checkpoint)
        print(f"\n🔄 تم العثور على تدريب سابق عند Epoch {latest_epoch}")
        print(f"   • المسار: {resume_path}")

        # تحميل النموذج
        model = YOLO(resume_path)
        start_epoch = latest_epoch
    else:
        start_epoch = 0
else:
    start_epoch = 0
    print(f"\n🆕 بدء تدريب جديد من Epoch 0")

# إعداد callback للحفظ الدوري
class SaveCallback:
    def __init__(self, save_interval=50, project_dir='/content/merged_training'):
        self.save_interval = save_interval
        self.project_dir = project_dir
        self.last_save = 0

    def on_epoch_end(self, trainer):
        epoch = trainer.epoch

        # حفظ كل save_interval Epoch
        if epoch >= self.last_save + self.save_interval:
            print(f"\n📦 جاري حفظ checkpoint عند Epoch {epoch}...")

            # تقييم النموذج قبل الحفظ
            metrics = trainer.model.val(data=yaml_path, split='val')

            # حفظ النموذج
            timestamp = datetime.now().strftime("%Y%m%d_%H%M")
            checkpoint_name = f'yolov8_merged_epoch{epoch}_{timestamp}.pt'
            checkpoint_path = os.path.join(self.project_dir, checkpoint_name)
            trainer.model.save(checkpoint_path)

            print(f"✅ تم حفظ النموذج: {checkpoint_path}")
            print(f"   • mAP50: {metrics.box.map50:.4f}")

            # حفظ المقاييس
            metrics_file = os.path.join(self.project_dir, f'metrics_epoch{epoch}_{timestamp}.txt')
            with open(metrics_file, 'w') as f:
                f.write(f"Epoch: {epoch}\n")
                f.write(f"mAP50: {metrics.box.map50:.4f}\n")
                f.write(f"mAP50-95: {metrics.box.map:.4f}\n")
                f.write(f"Precision: {metrics.box.mp:.4f}\n")
                f.write(f"Recall: {metrics.box.mr:.4f}\n")

            self.last_save = epoch

# بدء التدريب
print(f"\n⏳ بدء التدريب من Epoch {start_epoch} إلى 150...")

try:
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=640,
        batch=batch_size,
        device=0 if torch.cuda.is_available() else 'cpu',
        workers=4,
        cache=True,
        patience=20,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.4,
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        shear=2.0,
        perspective=0.001,
        mosaic=1.0,
        mixup=0.2,
        fliplr=0.5,
        close_mosaic=10,
        optimizer='AdamW',
        lr0=0.0005,
        weight_decay=0.0005,
        warmup_epochs=3,
        box=7.5,
        cls=0.5,
        dfl=1.5,
        project=project_dir,
        name='yolov8_merged',
        exist_ok=True,
        verbose=True,
        seed=42,
        resume=bool(resume_path)  # استئناف إذا وجد checkpoint
    )

    print("\n✅ اكتمل التدريب بنجاح!")

except KeyboardInterrupt:
    print("\n⚠️ تم إيقاف التدريب يدوياً")

    # حفظ النموذج الحالي
    current_epoch = start_epoch + 50  # تقديري
    checkpoint_path = save_model_checkpoint(model, f"{current_epoch}_interrupted")
    print(f"✅ تم حفظ النموذج عند الإيقاف: {checkpoint_path}")

except Exception as e:
    print(f"\n❌ خطأ في التدريب: {e}")

    # حفظ النموذج في حالة الخطأ
    checkpoint_path = save_model_checkpoint(model, "error")
    print(f"✅ تم حفظ النموذج قبل الخطأ: {checkpoint_path}")

In [ ]:
# ============================================
# الخلية 11: تقييم جميع النماذج المحفوظة
# ============================================

import pandas as pd
from IPython.display import display

print("="*60)
print("📊 تقييم جميع النماذج المحفوظة")
print("="*60)

# البحث عن جميع النماذج
checkpoints = [f for f in os.listdir(project_dir) if f.startswith('yolov8_merged_epoch') and f.endswith('.pt')]

if not checkpoints:
    print("❌ لا توجد نماذج محفوظة")
else:
    # ترتيب حسب Epoch
    results_list = []

    for checkpoint in sorted(checkpoints):
        try:
            # استخراج رقم Epoch من الاسم
            epoch = checkpoint.split('epoch')[1].split('_')[0]

            # تحميل النموذج
            model_path = os.path.join(project_dir, checkpoint)
            model = YOLO(model_path)

            print(f"\n📊 تقييم {checkpoint}...")

            # تقييم على validation set
            metrics = model.val(data=yaml_path, split='val', verbose=False)

            results_list.append({
                'Epoch': epoch,
                'Model': checkpoint,
                'mAP50': f"{metrics.box.map50:.4f}",
                'mAP50-95': f"{metrics.box.map:.4f}",
                'Precision': f"{metrics.box.mp:.4f}",
                'Recall': f"{metrics.box.mr:.4f}",
                'Size_MB': f"{os.path.getsize(model_path) / 1e6:.1f}"
            })

        except Exception as e:
            print(f"⚠️ خطأ في تقييم {checkpoint}: {e}")

    if results_list:
        # عرض النتائج كجدول
        df = pd.DataFrame(results_list)
        print("\n📈 نتائج تقييم جميع النماذج:")
        display(df)

        # حفظ النتائج
        df.to_csv(os.path.join(project_dir, 'evaluation_results.csv'), index=False)
        print(f"\n✅ تم حفظ النتائج في: {os.path.join(project_dir, 'evaluation_results.csv')}")

In [ ]:
# ============================================
# الخلية 12: استخراج أفضل نموذج
# ============================================

print("="*60)
print("🏆 أفضل نموذج")
print("="*60)

# تقييم جميع النماذج وإيجاد الأفضل
best_model = None
best_mAP50 = 0
best_epoch = None

for checkpoint in checkpoints:
    try:
        epoch = checkpoint.split('epoch')[1].split('_')[0]
        model_path = os.path.join(project_dir, checkpoint)

        # تقييم سريع
        model = YOLO(model_path)
        metrics = model.val(data=yaml_path, split='val', verbose=False)

        if metrics.box.map50 > best_mAP50:
            best_mAP50 = metrics.box.map50
            best_model = checkpoint
            best_epoch = epoch

    except:
        continue

if best_model:
    print(f"\n✅ أفضل نموذج:")
    print(f"   • الملف: {best_model}")
    print(f"   • Epoch: {best_epoch}")
    print(f"   • mAP50: {best_mAP50:.4f}")

    # نسخ أفضل نموذج إلى مجلد منفصل
    best_src = os.path.join(project_dir, best_model)
    best_dst = os.path.join(project_dir, 'best_model.pt')
    !cp "{best_src}" "{best_dst}"

    print(f"\n📦 تم نسخ أفضل نموذج إلى: {best_dst}")

    # تقييم نهائي على test set
    print(f"\n📊 تقييم أفضل نموذج على test set:")
    best = YOLO(best_dst)
    final_metrics = best.val(data=yaml_path, split='test')

    print(f"\n" + "="*60)
    print(f"📈 النتائج النهائية لأفضل نموذج:")
    print("="*60)
    print(f"mAP50:     {final_metrics.box.map50:.4f}")
    print(f"mAP50-95:  {final_metrics.box.map:.4f}")
    print(f"Precision: {final_metrics.box.mp:.4f}")
    print(f"Recall:    {final_metrics.box.mr:.4f}")
else:
    print("❌ لا توجد نماذج للتقييم")

In [13]:
# ============================================
# الخلية 10: تدريب YOLO على المجموعة المدمجة
# ============================================

print("="*60)
print("🚀 بدء تدريب YOLOv8 على المجموعة المدمجة")
print("="*60)

# معلومات GPU
if torch.cuda.is_available():
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   • الذاكرة: {total_memory:.1f} GB")

    # اختيار النموذج حسب الذاكرة
    if total_memory > 20:
        model = YOLO('yolov8x.pt')
        batch_size = 32
        print(f"✅ استخدام YOLOv8x")
    else:
        model = YOLO('yolov8m.pt')
        batch_size = 16
        print(f"✅ استخدام YOLOv8m")
else:
    model = YOLO('yolov8n.pt')
    batch_size = 8
    print(f"⚠️ استخدام YOLOv8n على CPU")

print(f"\n⚙️ إعدادات التدريب:")
print(f"   • حجم الدفعة: {batch_size}")
print(f"   • حجم الصورة: 640")
print(f"   • عدد الفئات: {num_classes}")

# بدء التدريب
results = model.train(
    data=yaml_path,
    epochs=80,
    imgsz=640,
    batch=batch_size,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    cache=True,
    patience=15,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.001,
    mosaic=1.0,
    mixup=0.2,
    fliplr=0.5,
    close_mosaic=10,
    optimizer='AdamW',
    lr0=0.0005,
    weight_decay=0.0005,
    warmup_epochs=3,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    project='/content/merged_training',
    name='yolov8_merged',
    exist_ok=True,
    verbose=True,
    seed=42
)

print("\n✅ التدريب قيد التشغيل!")

🚀 بدء تدريب YOLOv8 على المجموعة المدمجة
⚠️ استخدام YOLOv8n على CPU

⚙️ إعدادات التدريب:
   • حجم الدفعة: 8
   • حجم الصورة: 640
   • عدد الفئات: 58
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged_dataset.yaml, degrees=10.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.8, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8n.pt, momentum=0.937, mos

KeyboardInterrupt: 

In [ ]:
# ============================================
# الخلية 11: تقييم النموذج على بيانات الاختبار
# ============================================

from IPython.display import Image as IPImage

print("="*60)
print("📊 تقييم النموذج على بيانات الاختبار")
print("="*60)

# البحث عن أفضل نموذج
best_model_path = '/content/merged_training/yolov8_merged/weights/best.pt'

if os.path.exists(best_model_path):
    print(f"\n✅ تم العثور على أفضل نموذج: {best_model_path}")

    # تحميل أفضل نموذج
    best_model = YOLO(best_model_path)

    # تقييم على بيانات الاختبار
    metrics = best_model.val(
        data=yaml_path,
        split='test',
        batch=16,
        imgsz=640,
        conf=0.25,
        iou=0.5,
        device=0 if torch.cuda.is_available() else 'cpu'
    )

    print("\n" + "="*60)
    print("📈 نتائج التقييم على المجموعة المدمجة")
    print("="*60)
    print(f"mAP50:     {metrics.box.map50:.4f}")
    print(f"mAP50-95:  {metrics.box.map:.4f}")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall:    {metrics.box.mr:.4f}")

    # حساب F1-score
    if metrics.box.mp + metrics.box.mr > 0:
        f1 = 2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr)
        print(f"F1-score:  {f1:.4f}")

    # عرض مصفوفة الارتباك
    confusion_path = '/content/merged_training/yolov8_merged/confusion_matrix.png'
    if os.path.exists(confusion_path):
        print("\n📊 مصفوفة الارتباك:")
        display(IPImage(confusion_path))

    # عرض منحنيات التدريب
    results_img = '/content/merged_training/yolov8_merged/results.png'
    if os.path.exists(results_img):
        print("\n📈 منحنيات التدريب:")
        display(IPImage(results_img))

else:
    print("⚠️ النموذج غير موجود بعد. أكمل التدريب أولاً.")